In [ ]:
import sys

sys.path.append("..")
from core.DataLoader import (
    DataPreprocessor,
    get_load_config_from_yaml,
)
from importlib import reload
import keras
import core.keras_models as regression_transformer
import core.utils as utils
import os

PLOTS_DIR = f"plots/regression_transformer/"
MODEL_DIR = f"models/regression_transformer/"
CONFIG_PATH = "../config/load_nominal_config.yaml"

if not os.path.exists(PLOTS_DIR):
    os.makedirs(PLOTS_DIR)
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)


load_config = get_load_config_from_yaml(CONFIG_PATH)

DataProcessor = DataPreprocessor(load_config)


data_config = DataProcessor.load_from_npz(
    load_config.data_path, max_events=4_000_000, event_numbers="even"
)

X, y = DataProcessor.get_data()
del DataProcessor

In [ ]:
reload(regression_transformer)
Transformer = regression_transformer.CompactReconstructor(
    data_config, name="Transformer"
)

In [ ]:
Transformer.build_model(
    hidden_dim=128,
    num_layers=6,
    dropout_rate=0.2,
    use_global_event_inputs=True,
    log_variables=True,
)

In [ ]:
Transformer.adapt_normalization_layers(X)
Transformer.compile_model(
    loss={
        "assignment": utils.AssignmentLoss(),
        "normalized_regression": utils.RegressionLoss(),
    },
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4),
    metrics={
        "assignment": [utils.AssignmentAccuracy(name="accuracy")],
    },
)

In [ ]:
X_train, y_train, sample_weights = Transformer.prepare_training_data(
    X, y, sample_weights=Transformer.compute_sample_weights(X)
)

In [ ]:
Transformer.train_model(
    epochs=10,
    X=X_train,
    y=y_train,
    sample_weight=sample_weights,
    batch_size=1024,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            verbose=1,
            mode="min",
            min_lr=1e-6,
        ),
        keras.callbacks.TerminateOnNaN(),
    ],
    validation_split=0.1,
)

In [ ]:
Transformer.export_to_onnx(onnx_file_path=MODEL_DIR + "odd_model.onnx")
Transformer.export_to_onnx(onnx_file_path=MODEL_DIR + "even_model.onnx")


In [ ]:
Transformer.save_model(MODEL_DIR + "odd_model.keras")